# 模型评估实践

## 目标

掌握模型评估的实用技术：

- 交叉验证（KFold、StratifiedKFold、Leave-One-Out）
- 学习曲线（判断欠拟合/过拟合）
- 验证曲线（超参数调优）

## 1. 数据准备

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.model_selection import (
    KFold, StratifiedKFold, LeaveOneOut,
    cross_val_score, cross_validate,
    learning_curve, validation_curve,
    train_test_split
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子
np.random.seed(42)

print("导入完成！")

## 2. 准备真实数据集

我们使用乳腺癌数据集进行演示。

In [ ]:
# 加载乳腺癌数据集
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
target_names = data.target_names

print(f"数据集信息：")
print(f"样本数：{X.shape[0]}")
print(f"特征数：{X.shape[1]}")
print(f"类别分布：{np.bincount(y)}")
print(f"类别名称：{target_names}")

# 查看前几个特征
print(f"\n前5个特征名称：{feature_names[:5]}")

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"训练集大小：{X_train.shape[0]}")
print(f"测试集大小：{X_test.shape[0]}")
print(f"训练集类别分布：{np.bincount(y_train)}")
print(f"测试集类别分布：{np.bincount(y_test)}")

## 3. 交叉验证

交叉验证是评估模型泛化能力的重要方法。

### 3.1 K-Fold 交叉验证

In [ ]:
# 创建模型
model = LogisticRegression(max_iter=1000, random_state=42)

# 创建5折交叉验证
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 执行交叉验证
cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='accuracy')

print("K-Fold 交叉验证结果：")
print(f"各折准确率：{cv_scores}")
print(f"平均准确率：{cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# 可视化各折得分
plt.figure(figsize=(10, 4))
plt.bar(range(1, 6), cv_scores, alpha=0.7, color='skyblue')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'平均值: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('准确率')
plt.title('5折交叉验证各折准确率')
plt.ylim([cv_scores.min() - 0.05, cv_scores.max() + 0.05])
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 3.2 Stratified K-Fold 交叉验证

分层交叉验证确保每个fold中类别的分布与整体数据集一致。

In [ ]:
# 创建分层5折交叉验证
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 执行分层交叉验证
skf_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy')

print("Stratified K-Fold 交叉验证结果：")
print(f"各折准确率：{skf_scores}")
print(f"平均准确率：{skf_scores.mean():.4f} ± {skf_scores.std():.4f}")

# 对比 K-Fold 和 Stratified K-Fold
print("\n对比 K-Fold 和 Stratified K-Fold：")
print(f"K-Fold:              {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Stratified K-Fold:   {skf_scores.mean():.4f} ± {skf_scores.std():.4f}")

### 3.3 Leave-One-Out 交叉验证

留一法交叉验证，每次只用一个样本作为测试集。适合小数据集。

In [ ]:
# 创建留一交叉验证（在小数据集上演示）
loo = LeaveOneOut()

# 使用较小的数据集演示
X_small = X_train[:100]
y_small = y_train[:100]

loo_scores = cross_val_score(model, X_small, y_small, cv=loo, scoring='accuracy')

print(f"Leave-One-Out 交叉验证结果（100个样本）：")
print(f"准确率：{loo_scores.mean():.4f} ± {loo_scores.std():.4f}")
print(f"总fold数：{len(loo_scores)}")

### 3.4 多指标交叉验证

使用 `cross_validate` 可以同时评估多个指标。

In [ ]:
# 定义评估指标
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision_macro',
    'recall': 'recall_macro',
    'f1': 'f1_macro'
}

# 执行多指标交叉验证
cv_results = cross_validate(model, X_train, y_train, cv=5, scoring=scoring, return_train_score=True)

# 整理结果
metrics_summary = pd.DataFrame({
    '指标': ['准确率', '精确率', '召回率', 'F1分数'],
    '测试集均值': [
        cv_results['test_accuracy'].mean(),
        cv_results['test_precision'].mean(),
        cv_results['test_recall'].mean(),
        cv_results['test_f1'].mean()
    ],
    '测试集标准差': [
        cv_results['test_accuracy'].std(),
        cv_results['test_precision'].std(),
        cv_results['test_recall'].std(),
        cv_results['test_f1'].std()
    ],
    '训练集均值': [
        cv_results['train_accuracy'].mean(),
        cv_results['train_precision'].mean(),
        cv_results['train_recall'].mean(),
        cv_results['train_f1'].mean()
    ]
})

print("多指标交叉验证结果：")
print(metrics_summary.to_string(index=False))

In [ ]:
# 可视化多指标结果
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['accuracy', 'precision', 'recall', 'f1']
metric_names = ['准确率', '精确率', '召回率', 'F1分数']

for ax, metric, name in zip(axes.flat, metrics, metric_names):
    train_scores = cv_results[f'train_{metric}']
    test_scores = cv_results[f'test_{metric}']
    
    x = np.arange(5)
    width = 0.35
    
    ax.bar(x - width/2, train_scores, width, label='训练集', alpha=0.8)
    ax.bar(x + width/2, test_scores, width, label='测试集', alpha=0.8)
    
    ax.set_xlabel('Fold')
    ax.set_ylabel(name)
    ax.set_title(f'{name} - 训练集 vs 测试集')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.5 不同模型的交叉验证对比

In [ ]:
# 定义多个模型
models = {
    '逻辑回归': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    '支持向量机': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(random_state=42))
    ]),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42)
}

# 对每个模型进行交叉验证
results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = {
        'scores': cv_scores,
        'mean': cv_scores.mean(),
        'std': cv_scores.std()
    }
    print(f"{name}: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# 可视化模型对比
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(models))
width = 0.35

means = [results[name]['mean'] for name in models.keys()]
stds = [results[name]['std'] for name in models.keys()]

bars = ax.bar(x, means, width, yerr=stds, alpha=0.8, capsize=5)

# 添加数值标签
for bar, mean in zip(bars, means):
    height = bar.get_height()
    ax.annotate(f'{mean:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom')

ax.set_xlabel('模型')
ax.set_ylabel('准确率')
ax.set_title('不同模型的交叉验证准确率对比')
ax.set_xticks(x)
ax.set_xticklabels(models.keys())
ax.set_ylim([0.9, 1.0])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. 学习曲线

学习曲线帮助我们判断模型是否欠拟合或过拟合。

### 4.1 创建学习曲线函数

In [ ]:
def plot_learning_curve(estimator, X, y, title, cv=5):
    """
    绘制学习曲线
    
    参数:
        estimator: 模型
        X: 特征数据
        y: 目标变量
        title: 图表标题
        cv: 交叉验证折数
    """
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv,
        train_sizes=np.linspace(0.1, 1.0, 10),
        n_jobs=-1
    )
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)
    
    plt.figure(figsize=(10, 6))
    
    plt.plot(train_sizes, train_mean, 'o-', color='blue', label='训练集准确率')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                     alpha=0.2, color='blue')
    
    plt.plot(train_sizes, test_mean, 'o-', color='green', label='验证集准确率')
    plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, 
                     alpha=0.2, color='green')
    
    plt.xlabel('训练样本数')
    plt.ylabel('准确率')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # 返回诊断信息
    final_train_score = train_mean[-1]
    final_test_score = test_mean[-1]
    gap = final_train_score - final_test_score
    
    print(f"\n学习曲线诊断信息：")
    print(f"最终训练集准确率：{final_train_score:.4f}")
    print(f"最终验证集准确率：{final_test_score:.4f}")
    print(f"训练集和验证集差距：{gap:.4f}")
    
    # 诊断
    if final_train_score > 0.95 and final_test_score < 0.8:
        print("诊断：可能存在过拟合（模型太复杂）")
    elif final_train_score < 0.8 and final_test_score < 0.8:
        print("诊断：可能存在欠拟合（模型太简单）")
    elif gap > 0.1:
        print("诊断：存在过拟合，可以考虑增加数据或简化模型")
    else:
        print("诊断：模型拟合良好")
    
    return train_sizes, train_scores, test_scores

### 4.2 逻辑回归的学习曲线

In [ ]:
# 逻辑回归学习曲线
logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

plot_learning_curve(logistic_model, X_train, y_train, 
                    '逻辑回归学习曲线', cv=5)

### 4.3 随机森林的学习曲线

In [ ]:
# 随机森林学习曲线
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

plot_learning_curve(rf_model, X_train, y_train, 
                    '随机森林学习曲线', cv=5)

### 4.4 对比不同模型的学习曲线

In [ ]:
# 创建更简单和更复杂的模型进行对比
simple_model = LogisticRegression(max_iter=1000, C=0.01, random_state=42)  # 简单模型
complex_model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42)  # 复杂模型

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, model, title in zip(axes, [simple_model, complex_model], 
                              ['简单模型 (Logistic Regression, C=0.01)', 
                               '复杂模型 (Random Forest, 200 trees)']):
    train_sizes, train_scores, test_scores = learning_curve(
        model, X_train, y_train, cv=5,
        train_sizes=np.linspace(0.1, 1.0, 10)
    )
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)
    
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='训练集')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                   alpha=0.2, color='blue')
    ax.plot(train_sizes, test_mean, 'o-', color='green', label='验证集')
    ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, 
                   alpha=0.2, color='green')
    
    ax.set_xlabel('训练样本数')
    ax.set_ylabel('准确率')
    ax.set_title(title)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    
    # 添加诊断注释
    gap = train_mean[-1] - test_mean[-1]
    if gap > 0.1:
        ax.text(0.05, 0.95, '可能过拟合', transform=ax.transAxes,
               fontsize=10, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    elif train_mean[-1] < 0.8:
        ax.text(0.05, 0.95, '可能欠拟合', transform=ax.transAxes,
               fontsize=10, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

plt.tight_layout()
plt.show()

## 5. 验证曲线

验证曲线帮助我们选择最佳的超参数。

### 5.1 逻辑回归的正则化参数 C

In [ ]:
# 创建带标准化的逻辑回归
logistic_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

# 定义C参数范围（对数尺度）
param_range = np.logspace(-4, 4, 20)

# 绘制验证曲线
train_scores, test_scores = validation_curve(
    logistic_pipe, X_train, y_train, param_name='clf__C',
    param_range=param_range, cv=5, scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(12, 6))

plt.semilogx(param_range, train_mean, 'o-', color='blue', label='训练集准确率')
plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, 
                 alpha=0.2, color='blue')

plt.semilogx(param_range, test_mean, 'o-', color='green', label='验证集准确率')
plt.fill_between(param_range, test_mean - test_std, test_mean + test_std, 
                 alpha=0.2, color='green')

# 标记最佳参数
best_idx = np.argmax(test_mean)
best_c = param_range[best_idx]
best_score = test_mean[best_idx]
plt.scatter([best_c], [best_score], s=200, c='red', marker='*', zorder=5,
           label=f'最佳 C={best_c:.4f}')

plt.xlabel('正则化参数 C (对数尺度)')
plt.ylabel('准确率')
plt.title('逻辑回归：正则化参数 C 的验证曲线')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n验证曲线分析：")
print(f"最佳 C 值：{best_c:.4f}")
print(f"最佳验证集准确率：{best_score:.4f}")
print(f"\n参数说明：")
print(f"- C 值小：强正则化（模型简单，可能欠拟合）")
print(f"- C 值大：弱正则化（模型复杂，可能过拟合）")

### 5.2 随机森林的树数量 (n_estimators)

In [ ]:
# 随机森林的树数量验证曲线
rf_model = RandomForestClassifier(random_state=42)

# 定义n_estimators参数范围
param_range = np.arange(10, 310, 20)

# 绘制验证曲线
train_scores, test_scores = validation_curve(
    rf_model, X_train, y_train, param_name='n_estimators',
    param_range=param_range, cv=5, scoring='accuracy', n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(12, 6))

plt.plot(param_range, train_mean, 'o-', color='blue', label='训练集准确率')
plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, 
                 alpha=0.2, color='blue')

plt.plot(param_range, test_mean, 'o-', color='green', label='验证集准确率')
plt.fill_between(param_range, test_mean - test_std, test_mean + test_std, 
                 alpha=0.2, color='green')

# 标记最佳参数
best_idx = np.argmax(test_mean)
best_n = param_range[best_idx]
best_score = test_mean[best_idx]
plt.scatter([best_n], [best_score], s=200, c='red', marker='*', zorder=5,
           label=f'最佳 n_estimators={best_n}')

plt.xlabel('树数量 (n_estimators)')
plt.ylabel('准确率')
plt.title('随机森林：树数量的验证曲线')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n验证曲线分析：")
print(f"最佳树数量：{best_n}")
print(f"最佳验证集准确率：{best_score:.4f}")

### 5.3 随机森林的最大深度 (max_depth)

In [ ]:
# 随机森林的最大深度验证曲线
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 定义max_depth参数范围（None表示无限制）
param_range = [1, 2, 3, 5, 8, 10, 15, 20, None]
param_labels = [str(p) if p is not None else 'None' for p in param_range]

# 绘制验证曲线
train_scores, test_scores = validation_curve(
    rf_model, X_train, y_train, param_name='max_depth',
    param_range=param_range, cv=5, scoring='accuracy', n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(12, 6))

x_pos = np.arange(len(param_range))

plt.plot(x_pos, train_mean, 'o-', color='blue', label='训练集准确率')
plt.fill_between(x_pos, train_mean - train_std, train_mean + train_std, 
                 alpha=0.2, color='blue')

plt.plot(x_pos, test_mean, 'o-', color='green', label='验证集准确率')
plt.fill_between(x_pos, test_mean - test_std, test_mean + test_std, 
                 alpha=0.2, color='green')

# 标记最佳参数
best_idx = np.argmax(test_mean)
best_depth = param_range[best_idx]
best_score = test_mean[best_idx]
plt.scatter([best_idx], [best_score], s=200, c='red', marker='*', zorder=5,
           label=f'最佳 max_depth={param_labels[best_idx]}')

plt.xlabel('最大深度 (max_depth)')
plt.ylabel('准确率')
plt.title('随机森林：最大深度的验证曲线')
plt.xticks(x_pos, param_labels)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n验证曲线分析：")
print(f"最佳最大深度：{param_labels[best_idx]}")
print(f"最佳验证集准确率：{best_score:.4f}")

### 5.4 SVM 的核函数和 C 参数

In [ ]:
# SVM不同核函数的验证曲线
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 核函数对比
kernels = ['linear', 'rbf', 'poly']
kernel_scores = []

for kernel in kernels:
    svm_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel=kernel, random_state=42))
    ])
    scores = cross_val_score(svm_pipe, X_train, y_train, cv=5, scoring='accuracy')
    kernel_scores.append(scores)

kernel_mean = np.array([s.mean() for s in kernel_scores])
kernel_std = np.array([s.std() for s in kernel_scores])

axes[0].bar(kernels, kernel_mean, yerr=kernel_std, alpha=0.7, capsize=5)
axes[0].set_ylabel('准确率')
axes[0].set_title('SVM：不同核函数的准确率对比')
axes[0].grid(True, alpha=0.3, axis='y')

# RBF核的C参数验证曲线
param_range = np.logspace(-2, 2, 10)

train_scores, test_scores = validation_curve(
    Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', random_state=42))
    ]),
    X_train, y_train, param_name='svm__C',
    param_range=param_range, cv=5, scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

axes[1].semilogx(param_range, train_mean, 'o-', color='blue', label='训练集')
axes[1].fill_between(param_range, train_mean - train_std, train_mean + train_std, 
                   alpha=0.2, color='blue')
axes[1].semilogx(param_range, test_mean, 'o-', color='green', label='验证集')
axes[1].fill_between(param_range, test_mean - test_std, test_mean + test_std, 
                   alpha=0.2, color='green')

best_idx = np.argmax(test_mean)
best_c = param_range[best_idx]
best_score = test_mean[best_idx]
axes[1].scatter([best_c], [best_score], s=200, c='red', marker='*', zorder=5)

axes[1].set_xlabel('C 参数 (对数尺度)')
axes[1].set_ylabel('准确率')
axes[1].set_title('SVM (RBF核)：C 参数的验证曲线')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSVM 分析结果：")
print(f"最佳核函数：{kernels[np.argmax(kernel_mean)]}")
print(f"最佳 C 值 (RBF核)：{best_c:.4f}")

## 6. 综合实例：完整的模型评估流程

In [ ]:
# 综合实例：从交叉验证到模型选择

# 1. 定义候选模型
candidates = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', random_state=42))
    ]),
    'Random Forest': RandomForestClassifier(random_state=42)
}

# 2. 对每个模型进行交叉验证
cv_results = {}
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_results[name] = {
        'mean': scores.mean(),
        'std': scores.std(),
        'scores': scores
    }

# 3. 选择最佳模型
best_model_name = max(cv_results, key=lambda x: cv_results[x]['mean'])
best_model = candidates[best_model_name]
best_score = cv_results[best_model_name]['mean']

print("=" * 60)
print("模型评估结果")
print("=" * 60)
for name, result in cv_results.items():
    print(f"{name}: {result['mean']:.4f} ± {result['std']:.4f}")

print(f"\n最佳模型: {best_model_name} (准确率: {best_score:.4f})")
print("=" * 60)

In [ ]:
# 4. 绘制最佳模型的学习曲线
fig, ax = plt.subplots(figsize=(10, 6))

train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train, y_train, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

ax.plot(train_sizes, train_mean, 'o-', color='blue', label='训练集')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                alpha=0.2, color='blue')
ax.plot(train_sizes, test_mean, 'o-', color='green', label='验证集')
ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, 
                alpha=0.2, color='green')

ax.set_xlabel('训练样本数')
ax.set_ylabel('准确率')
ax.set_title(f'最佳模型学习曲线: {best_model_name}')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n学习曲线分析：")
print(f"最终训练集准确率：{train_mean[-1]:.4f}")
print(f"最终验证集准确率：{test_mean[-1]:.4f}")
print(f"差距：{train_mean[-1] - test_mean[-1]:.4f}")

In [ ]:
# 5. 在测试集上评估
best_model.fit(X_train, y_train)
test_score = best_model.score(X_test, y_test)

print(f"\n最终测试集准确率：{test_score:.4f}")
print(f"交叉验证准确率：{best_score:.4f}")

# 检查是否存在显著差异
if abs(test_score - best_score) > 0.05:
    print("⚠️  警告：测试集准确率与交叉验证准确率差异较大，可能存在数据泄露或过拟合")
else:
    print("✓ 模型泛化能力良好")

## 总结

### 交叉验证策略

| 策略 | 特点 | 适用场景 |
|------|------|----------|
| K-Fold | 将数据分成k份，轮流作为验证集 | 通用场景 |
| Stratified K-Fold | 保持类别分布 | 分类任务，特别是类别不平衡 |
| Leave-One-Out | 每次留1个样本测试 | 小数据集 |

### 学习曲线诊断

| 现象 | 诊断 | 解决方案 |
|------|------|----------|
| 训练集和验证集都低 | 欠拟合 | 增加模型复杂度、增加特征 |
| 训练集高、验证集低 | 过拟合 | 增加数据、正则化、简化模型 |
| 两者都高且接近 | 拟合良好 | 可以尝试进一步优化 |

### 验证曲线用途

- **超参数调优**：找到最佳参数值
- **理解模型行为**：观察参数如何影响性能
- **避免过拟合**：选择泛化能力最好的参数

### 实践建议

1. **始终使用交叉验证**：不要只依赖单次划分
2. **分层抽样**：对于分类任务，优先使用 Stratified K-Fold
3. **先看学习曲线**：了解当前模型状态（欠拟合/过拟合）
4. **再用验证曲线**：针对性地调整超参数
5. **验证集测试**：最后用独立的测试集验证泛化能力